# Fine-tune a vision-language model to read the registry's handwriting

**What this does:** teaches **Qwen3-VL-8B** (a vision-language model) to read one field of the maternal registry at a time,
with the field's context (label, expected format), from crops exported by `python -m tools.export_vlm_dataset` (specimen pages) and
`python -m tools.synth_midwife` (synthetic crops written in the real midwife's formats: cmHg "11/7", "16SA+3j", "NF", "Reçu", "nég", "colorées"…). Upload `vlm_dataset_v2.zip` (both).

1. measures the base model on held-out pages (patients 8–10, never trained on);
2. fine-tunes it with QLoRA (4-bit model + small trainable adapters: fits Colab's free **T4** GPU);
3. measures it again, per field type and per photo quality;
4. downloads the adapter (`vlm_ocr_lora.zip`, ~100–200 MB) to run on a GPU laptop (RTX 4060 8 GB), fully offline.

**Data rule (from the challenge brief): no real patient data may be sent to a third-party service.**
Upload only `vlm_dataset.zip` (the organisers' *synthetic* specimen pages). The real booklet photos
(`vlm_dataset_real/`) stay on the team's machines: use section 7 when running this notebook locally (Jupyter).

**Runtime → Change runtime type → T4 GPU**, then *Run all*. Training takes about 30–60 min.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install

In [ ]:
%%capture
!pip install unsloth rapidfuzz

## 2. Settings
`MODEL`: the default is the strongest that trains on a free T4. If it runs out of memory, use the 7B or 3B line.

In [ ]:
MODEL = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"     # strongest that fits a T4 in 4-bit
# MODEL = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"         # fallback
# MODEL = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"         # small: also trains on an 8 GB laptop GPU

START_FROM_ADAPTER = True   # continue from the first run's vlm_ocr_lora.zip (upload it); False = start from the base model
SYNTH_SHARE = 0.5          # share of training examples written in the midwife's formats (the rest: specimen crops)
MAX_STEPS = 400          # x 8 examples per step; raise for a longer run (watch the val score)
LEARNING_RATE = 2e-4
LORA_RANK = 16
EVAL_N = 300             # held-out crops scored before/after (each takes ~0.5-1 s)
MAX_SIDE = 768           # crops are resized so their longest side is at most this (pixels)
SEED = 3407

## 3. Data
Upload `vlm_dataset.zip` (from `eval_out/` in the repo) when the file picker appears,
or put it in Google Drive and set `ZIP` to its path.

In [ ]:
import os, json, zipfile, random
ZIP = "vlm_dataset_v2.zip"   # or vlm_dataset.zip (specimen only)
if not os.path.exists(ZIP):
    from google.colab import files
    up = files.upload()
    ZIP = next(iter(up))
with zipfile.ZipFile(ZIP) as z:
    z.extractall(".")
ROOT = "vlm_dataset"

def load(split):
    path = f"{ROOT}/{split}.jsonl"
    return [json.loads(l) for l in open(path, encoding="utf-8")] if os.path.exists(path) else []

train_rows, val_rows = load("train"), load("val")
synth_train, synth_val = load("synth_train"), load("synth_val")
print(len(train_rows), "specimen train crops,", len(val_rows), "held-out;", len(synth_train), "midwife-format train,", len(synth_val), "held-out")
print(train_rows[0]["prompt"][:200], "...", "->", train_rows[0]["answer"])

In [ ]:
from PIL import Image

def image(row, root=ROOT):
    img = Image.open(f"{root}/{row['image']}").convert("RGB")
    s = MAX_SIDE / max(img.size)
    return img.resize((max(28, int(img.width * s)), max(28, int(img.height * s)))) if s < 1 else img

def conversation(row, root=ROOT):
    return {"messages": [
        {"role": "user", "content": [{"type": "image", "image": image(row, root)}, {"type": "text", "text": row["prompt"]}]},
        {"role": "assistant", "content": [{"type": "text", "text": row["answer"]}]},
    ]}

random.seed(SEED)
n_total = MAX_STEPS * 8
n_synth = int(n_total * SYNTH_SHARE) if synth_train else 0
mix = random.sample(synth_train, min(n_synth, len(synth_train))) + random.sample(train_rows, min(n_total - n_synth, len(train_rows)))
random.shuffle(mix)
train_data = [conversation(r) for r in mix]
print(len(train_data), "training examples,", sum(r["variant"] == "synth_midwife" for r in mix), "in the midwife's formats")
# held-out samples: specimen pages (clean / simulated photos) and midwife-format crops
eval_rows = random.sample(val_rows, min(EVAL_N, len(val_rows)))
eval_synth = random.sample(synth_val, min(EVAL_N // 2, len(synth_val))) if synth_val else []

## 4. Load the model and measure it before training

In [ ]:
import torch, os, zipfile
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(MODEL, load_in_4bit=True, use_gradient_checkpointing="unsloth")
# trainable adapters (LoRA); they start at zero, so the model behaves like the base model until trained
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,      # handwriting is a visual problem: adapt the vision encoder too
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_RANK, lora_alpha=LORA_RANK, lora_dropout=0, bias="none", random_state=SEED,
)

def load_adapter(zip_name, folder):
    """Copy the weights of a saved adapter (same settings) into the model."""
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    if not os.path.exists(folder):
        if not os.path.exists(zip_name):
            from google.colab import files
            print(f"Upload {zip_name}")
            files.upload()
        zipfile.ZipFile(zip_name).extractall(".")
    res = set_peft_model_state_dict(model, load_file(f"{folder}/adapter_model.safetensors"))
    print(f"loaded {folder}; unexpected keys: {len(getattr(res, 'unexpected_keys', []) or [])}")

if START_FROM_ADAPTER:
    load_adapter("vlm_ocr_lora.zip", "vlm_ocr_lora")   # the first run: continue from it


In [ ]:
import re
from collections import defaultdict
from rapidfuzz.distance import Levenshtein

def ask(img, prompt, max_new_tokens=24):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

norm = lambda t: re.sub(r"\s", "", t or "").lower()

def score(rows, root=ROOT, show=8):
    """exact match and character similarity (1 = perfect), overall, per field type and per photo quality"""
    FastVisionModel.for_inference(model)
    res = []
    for i, r in enumerate(rows):
        got = ask(image(r, root), r["prompt"])
        sim = Levenshtein.normalized_similarity(norm(got), norm(r["answer"]))
        res.append((r, got, sim))
        if i < show:
            print(f"  {r['key'][-28:]:28s} truth={r['answer']!r:16s} read={got!r}")
    def line(name, rs):
        return f"{name:22s} n={len(rs):4d}  exact {sum(s == 1 for _, _, s in rs) / len(rs):6.1%}  similarity {sum(s for _, _, s in rs) / len(rs):.3f}"
    print(line("ALL", res))
    for group in ("variant", "dtype"):
        by = defaultdict(list)
        for x in res:
            by[x[0][group]].append(x)
        for k in sorted(by):
            print("  " + line(f"{group}={k}", by[k]))
    return res

print("BEFORE (held-out specimen patients 8-10):")
before = score(eval_rows)
if eval_synth:
    print("\nBEFORE (held-out crops in the midwife's formats):")
    before_synth = score(eval_synth, show=4)

## 5. Fine-tune (QLoRA)

In [ ]:
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# (the adapters were attached in section 4)
FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_data,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir="outputs",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=2048,
    ),
)
stats = trainer.train()
print(f"trained {MAX_STEPS} steps in {stats.metrics['train_runtime'] / 60:.0f} min")

In [ ]:
print("AFTER (same held-out specimen crops):")
after = score(eval_rows)
b = sum(s for *_, s in before) / len(before); a = sum(s for *_, s in after) / len(after)
print(f"\nspecimen: similarity {b:.3f} -> {a:.3f} | exact {sum(s == 1 for *_, s in before)} -> {sum(s == 1 for *_, s in after)} of {len(after)}")
if eval_synth:
    print("\nAFTER (same held-out crops in the midwife's formats):")
    after_synth = score(eval_synth, show=4)
    b = sum(s for *_, s in before_synth) / len(before_synth); a = sum(s for *_, s in after_synth) / len(after_synth)
    print(f"\nmidwife formats: similarity {b:.3f} -> {a:.3f} | exact {sum(s == 1 for *_, s in before_synth)} -> {sum(s == 1 for *_, s in after_synth)} of {len(after_synth)}")


## 6. Save and download the adapter
The adapter is small (the base model is downloaded again from Hugging Face on the GPU laptop).

In [ ]:
model.save_pretrained("vlm_ocr_lora_v2")
tokenizer.save_pretrained("vlm_ocr_lora_v2")
json.dump({"base_model": MODEL, "max_steps": MAX_STEPS, "start_from_adapter": START_FROM_ADAPTER, "synth_share": SYNTH_SHARE, "lora_rank": LORA_RANK, "max_side": MAX_SIDE},
          open("vlm_ocr_lora_v2/training_info.json", "w"))
!zip -qr vlm_ocr_lora_v2.zip vlm_ocr_lora_v2
try:
    from google.colab import files
    files.download("vlm_ocr_lora_v2.zip")
except ImportError:
    print("saved vlm_ocr_lora_v2.zip")

## 7. Test on the real booklet photos

The 93 real crops (`vlm_dataset_real.zip`) come from the 5 booklet photos: field crops only, identifier areas
(name, ID, phone, address…) painted out before cutting. The brief forbids sending *real* patient data to a third
party; the organisers describe the dataset as synthetic. Running this on Colab is the team's decision: set
`ALLOW_REAL_IN_COLAB = True` to confirm it. Only scores are printed (no values), and before running, tick
Edit → Notebook settings → *Omit code cell output when saving*. Delete the files from the session afterwards (last cell).

If the session was restarted since training: set `START_FROM_ADAPTER = False`, run sections 1–4 (stop the BEFORE scoring with ■ if you like), then this section loads `vlm_ocr_lora_v2.zip` (or `vlm_ocr_lora.zip`).


In [ ]:
ALLOW_REAL_IN_COLAB = False   # team decision: set to True to run the real crops on Colab
REAL_ROOT = "vlm_dataset_real"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not ALLOW_REAL_IN_COLAB:
    print("Not running: set ALLOW_REAL_IN_COLAB = True to test the real crops on Colab.")
else:
    import os, zipfile, json
    if not os.path.exists(REAL_ROOT):
        if IN_COLAB:
            from google.colab import files
            print("Upload vlm_dataset_real.zip")
            up = files.upload()
            zipfile.ZipFile(next(iter(up))).extractall(".")
    if "model" not in globals():   # session restarted: run sections 1-4 first (with START_FROM_ADAPTER = False)
        raise RuntimeError("Run sections 1-4 first, then this cell loads the trained adapter.")
    if "trainer" not in globals():  # not trained in this session: load the latest saved adapter
        load_adapter("vlm_ocr_lora_v2.zip", "vlm_ocr_lora_v2") if os.path.exists("vlm_ocr_lora_v2.zip") or os.path.exists("vlm_ocr_lora_v2") \
            else load_adapter("vlm_ocr_lora.zip", "vlm_ocr_lora")
    real_rows = [json.loads(l) for l in open(f"{REAL_ROOT}/real.jsonl", encoding="utf-8")]
    print(len(real_rows), "real crops (current OCR gets about 66% of these fields right)")
    real_res = score(real_rows, root=REAL_ROOT, show=0)  # scores only: no patient values in the output


In [ ]:
# remove the real crops from the Colab session when done
import shutil, os
for p in ("vlm_dataset_real", "vlm_dataset_real.zip"):
    if os.path.isdir(p): shutil.rmtree(p)
    elif os.path.exists(p): os.remove(p)
print("real crops removed from this session")


## 8. Demo: the app's extractor with the trained model

Runs the real pipeline (page type, fields, identifiers masked, checkboxes) with the trained model reading
every handwritten value: `extract(photo, backend="vlm")`. **Use a fresh session** (Runtime → Restart
session), then run section 1 (install) and this section only: the model must not be loaded twice on a T4.
Upload a page photo: a specimen page (synthetic) is always fine; a real booklet photo only if the team
accepts it (see section 7).


In [ ]:
import os, subprocess, sys, zipfile
REPO_URL = "https://github.com/husseinassi1982-hue/CodeML2026.git"
ADAPTER_URL = "https://github.com/husseinassi1982-hue/CodeML2026/releases/download/vlm-ocr-v2/vlm_ocr_lora_v2.zip"
if not os.path.exists("CodeML2026"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidocr", "onnxruntime", "rapidfuzz", "pymupdf",
                "opencv-python-headless"], check=True)
os.chdir("CodeML2026"); sys.path.insert(0, os.getcwd())
if not os.path.exists("vlm_ocr_lora_v2.zip"):
    subprocess.run(["wget", "-q", ADAPTER_URL], check=True)
os.environ["DAYONE_VLM_ADAPTER"] = os.path.abspath("vlm_ocr_lora_v2.zip")

from google.colab import files
print("Upload a page photo")
photo = next(iter(files.upload()))

from extraction import extract
r = extract(photo, backend="vlm")
print(f"page: {r.page_type_label} | backend: {r.backend} | {r.processing_ms / 1000:.0f} s | {r.summary}")
for k, g in r.fields.items():
    if g.status.value != "NOT_PROVIDED":
        print(f"  {g.label_fr[:40]:40s} {g.status.value:14s} {str(g.display or g.value):18s} {g.confidence:.2f}")
